# 03 -- Word frequency

Most frequent words in r/litigi comments, without Italian and English stopwords (NLTK), and the contexts in which a word is used.

In [ ]:
import re
from pathlib import Path

import nltk
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.text import Text

from subreddit_lens import load_comments, preprocess
from subreddit_lens.constants import REMOVED_BODIES

from subreddit_lens import load_config

# Locate the analytics directory, which holds subreddit-lens.toml, so paths
# work regardless of the working directory.
ANALYTICS_DIR = next(
    p
    for p in [Path.cwd(), Path.cwd() / "analytics", *Path.cwd().parents]
    if (p / "subreddit-lens.toml").exists()
)
config = load_config(ANALYTICS_DIR / "subreddit-lens.toml")
DATA_DIR = config.data_dir
OUTPUT_DIR = config.output_dir
OUTPUT_DIR.mkdir(exist_ok=True)

# Stopword lists, downloaded once to the NLTK data directory.
nltk.download("stopwords", quiet=True)
%matplotlib inline

In [ ]:
stop_words = set(stopwords.words("italian")) | set(stopwords.words("english"))
# Fragments of URLs and Reddit markup.
stop_words |= {
    "com", "http", "https", "www", "reddit", "redd", "comments",
    "message", "savevideo", "download",
}
print(f"{len(stop_words)} stopwords")

In [ ]:
# Words made of letters only, accented letters included ('perché', 'più').
WORD_PATTERN = re.compile(r"[^\W\d_]+")


def tokenize(body: str) -> list[str]:
    """Split a comment into lower-case words, without stopwords and single letters."""
    words = WORD_PATTERN.findall(preprocess(body).lower())
    return [w for w in words if len(w) > 1 and w not in stop_words]


tokenize("Perché l'hai già detto più volte? [Qui](https://example.com) c'è tutto.")

In [ ]:
df = load_comments(DATA_DIR / "litigi_comments.parquet")
# Deleted and removed comments have no text.
df = df[~df["body"].isin(REMOVED_BODIES)]
df

## Word frequency

In [ ]:
frequency_distribution = FreqDist(w for body in df["body"] for w in tokenize(body))
print(frequency_distribution)

In [ ]:
frequency_distribution.most_common(100)

In [ ]:
frequency_distribution.plot(45, cumulative=False)

## Concordance

The contexts in which a word is used, stopwords included.

In [ ]:
WORD = "puledra"

matches = df.loc[df["body"].str.contains(WORD, case=False), "body"]
words = (w for body in matches for w in WORD_PATTERN.findall(preprocess(body).lower()))
Text(words).concordance(WORD, width=100, lines=20)